In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import nibabel as nib
import os
import cv2


In [ ]:
def load_patient_images(nb_patient=1, training_path="database/training", validation_path="database/validation"):
    # obtenir le nom des images dans le dossier sans les deux premiers fichiers
    images_names = os.listdir(f"{training_path}/patient{nb_patient:03d}")[2:]

    # charger les images
    image_4d = nib.load(f"{training_path}/patient{nb_patient:03d}/{images_names[0]}")
    diastole = nib.load(f"{training_path}/patient{nb_patient:03d}/{images_names[1]}")
    diastole_segmentation = nib.load(f"{training_path}/patient{nb_patient:03d}/{images_names[2]}")
    systole = nib.load(f"{training_path}/patient{nb_patient:03d}/{images_names[3]}")
    systole_segmentation = nib.load(f"{training_path}/patient{nb_patient:03d}/{images_names[4]}")

    return {"image_4d":image_4d.get_fdata(), "diastole":diastole.get_fdata(), "diastole_segmentation":diastole_segmentation.get_fdata(), "systole":systole.get_fdata(), "systole_segmentation":systole_segmentation.get_fdata()}


# im = load_patient_images(1)["diastole_segmentation"][:,:,4]
# show_image(im)
# im2 = load_patient_images(1)["diastole"][:,:,4]
# show_image(im2)

In [ ]:
def load_frame(nb_patient=1, zcoord="mid", image_type="diastole"):
    image = load_patient_images(nb_patient)[image_type]
    if zcoord == "mid":
        z = image.shape[2]//2
        frame = image[:,:,z]
    else: 
        try:
            z = int(zcoord)
            frame = image[:,:,z]
        except:
            print("zcoord should be an integer or 'mid'")
            return None
    return frame

In [ ]:
def gradient_magnitude(image):
    gradx = np.gradient(image, axis=0)
    grady = np.gradient(image, axis=1)
    magnitude = np.sqrt(gradx**2 + grady**2)
    return magnitude

In [ ]:
def seuil(image, threshold):
    image[image < threshold] = 0
    return image

In [ ]:
def diff_image(image1, image2, seuil=0):
    diff = np.abs(image1 - image2)
    diff[diff < seuil] = 0
    return diff

In [ ]:
def show_image(im, title=""):
    plt.imshow(im, cmap='gray')
    plt.title(title)
    plt.show()

In [ ]:
def hough_transform(im, param1=30, param2=30, minradius=0, maxRadius=0):
    im = np.uint8(im)
    hough = cv2.HoughCircles(im, cv2.HOUGH_GRADIENT, 1, 20, param1=param1, param2=param2, minRadius=minradius, maxRadius=maxRadius)
    return hough


In [ ]:
def normalize(im):
    im = im - np.min(im)
    im = im / (np.max(im)- np.min(im))
    return im

In [ ]:
def find_LV_whih_threshold(nb_person, seuil):

    diastole = load_frame(nb_person, "mid", "diastole")
    systole = load_frame(nb_person, "mid", "systole")

    show_image(diastole, "Diastole avant normalisation")
    
    show_image(systole, "Systole avant normalisation")
    diff = np.uint8(diff_image(diastole, systole, seuil=0))
    show_image(diff, "Difference between diastole and systole avant normalisation")

    diastole = normalize(diastole)
    systole = normalize(systole)

    show_image(diastole, "Diastole")

    show_image(systole, "Systole")

    diff = diff_image(diastole, systole, seuil=0)
    print
    show_image(diff, "Difference between diastole and systole after normalisation")

    diastole_circles = np.uint8(np.copy(diastole))
    diastole_circles = cv2.cvtColor(diastole_circles, cv2.COLOR_GRAY2BGR)

    # show_image(diff, "Magnitude of the gradient of the difference between diastole and systole")

    circles = hough_transform(diff, 20, 20, 0, 0)

    i = np.uint8(circles[0,0])

    cv2.circle(diastole_circles,(i[0],i[1]),i[2],(0,255,0),2)
        # draw the center of the circle
    cv2.circle(diastole_circles,(i[0],i[1]),2,(0,0,255),3)

    show_image(diastole_circles, "Circled image")

In [ ]:
def find_LV(diastole, systole, seuil=0):
    """
    Cette fonction renvoie le centre du cercle le plus probables sur la différence des images.
    """
    diff = diff_image(diastole, systole, seuil=seuil)
    diff = np.uint8(diff)
    # show_image(diff, "Difference between diastole and systole")
    diff = cv2.GaussianBlur(diff, (5, 5), 1)
    # show_image(diff, "Difference between diastole and systole")
    diff = cv2.Canny(diff, 50, 100)
    # show_image(diff, "Difference between diastole and systole")
    circles = hough_transform(diff, 1, 1, 10, 20)
    first_circle = np.uint8(circles[0][0])

    x, y, _ = first_circle
    return (x, y)

def draw_circle(im, center):
    if len(im.shape) == 2:  
        im = cv2.cvtColor(np.uint8(im), cv2.COLOR_GRAY2BGR)   

    cv2.circle(im,center,2,(0,0,255),3)
    
    return im




In [ ]:
def in_LV(center:tuple, im_answer:np.array):
    """
    Cette fonction renvoie True si le centre est dans le LV, False sinon.
    """
    if im_answer[center[1], center[0]] == 3:
        return True
    else:
        return False

In [ ]:
def LV_seed_accuracy(indexes, LV_function):
    """
    Cette fonction renvoie la précision de la fonction de détection de du LV.
    """
    results = np.zeros(len(indexes))
    
    for i, p in enumerate(indexes):
        im_answer = load_frame(p, "mid", "diastole_segmentation")
        diastole = load_frame(p, "mid", "diastole")
        systole = load_frame(p, "mid", "systole")
        center = LV_function(diastole, systole)

        if in_LV(center, im_answer):
            results[i] = 1
        else:
            results[i] = 0

    return float(np.mean(results))



In [ ]:
indexes = [i for i in range(95, 100)]
LV_accuracy(indexes, find_LV)

In [ ]:
import numpy as np
def mask_accuracy(mask, im_anwser):
    """
    Cette fonction renvoie le ratio de points bien et mal classés par le masque.
    Le ratio de points bien placés est compris entre 0 et 1 alors que le ratio de points mal placés peut dépasser 1.
    """

    good_mask_points_args = np.argwhere((mask > 0) & (im_answer == 3))
    wrong_mask_points_args = np.argwhere((mask > 0) & (im_answer != 3))
    LV_points_args = np.argwhere(im_answer == 3)
    good_points_ratio = np.size(good_mask_points_args) / np.size(LV_points_args)
    wrong_points_ratio = np.size(wrong_mask_points_args) / np.size(LV_points_args)
    return (good_points_ratio, wrong_points_ratio)



mask = np.array([[0,0,1,1,1,0,0,0],[0,0,1,1,1,0,0,0]])
im_answer = np.array([[0,0,3,0,0,0,0,0],[0,0,3,0,0,0,0,0]])
mask_accuracy(mask, im_answer)


In [ ]:
def growth_function_accuracy(indexes, LV_growth_function, find_LV_seed):
    """
    Cette fonction renvoie la précision de la fonction de détection de du LV.
    """
    results = []
    
    for i, p in enumerate(indexes):
        im_answer = load_frame(p, "mid", "diastole_segmentation")
        diastole = load_frame(p, "mid", "diastole")
        systole = load_frame(p, "mid", "systole")
        seed = find_LV_seed(diastole, systole)
        mask = LV_growth_function(diastole, systole, seed)

        if in_LV(seed, im_answer):
            good_points_ratio, wrong_points_ratio = mask_accuracy(mask, im_answer)
            results.append([good_points_ratio, wrong_points_ratio])
        results = np.array(results)
    good_seed_ratio = results[:,0].size / len(indexes)
    mean_good_points_ratio = np.mean(results[:,0])
    mean_wrong_points_ratio = np.mean(results[:,1])
    print(f"Ratio de graines bien placées: {good_seed_ratio}")
    print(f"Ratio des pixels du masque bien placés: {mean_good_points_ratio}?\nRatio des pixels du masque mal placés: {mean_wrong_points_ratio}")
    return (mean_good_points_ratio, mean_wrong_points_ratio)

In [ ]:
import gimp

In [ ]:
im = load_frame(1, "mid", "diastole_segmentation")

In [ ]:
def competition(i):

    diastole = load_frame(i, "mid", "diastole")
    systole = load_frame(i, "mid", "systole")

    # show_image(diastole, "Diastole")
    # show_image(systole, "Systole")

    new_diastole = np.copy(diastole) 

    circle = find_LV(diastole, systole)

    # for circle in circles[:1]:
    #     print(circle)
    x, y= circle
    new_diastole = draw_circle(new_diastole, (x, y))

    # x, y, radius = find_LV(diastole, systole)[0]

    # print(x, y, radius)

    # new_diastole = draw_circle(new_diastole, (x, y), radius)

    show_image(new_diastole, "Diastole with the circle {i}".format(i=i))



# competition(20)

for i in range(1, 4):
    competition(i)





In [ ]:
help(cv2.Canny)